# Evaluating Classifiers, Explained Simply
Accuracy, the confusion matrix, precision, recall, F1, thresholds and the ROC curve. Runs free in Google Colab.

Run each cell from top to bottom with **Shift + Enter**.

## 1. The data
A bank has 4,000 card transactions. About 3% are fraud. Facts: amount, whether it happened at night, distance from home, and whether the shop is new to this customer.

### 4,000 transactions
- `transactions.csv`: 4,000 transactions: amount in dollars, night_time (1 = yes), distance from home in km, new_merchant (1 = yes), fraud (1 = yes)
- `value_counts().to_dict()`: how many normal (0) and fraud (1) transactions

In [ ]:
import pandas as pd

url = ("https://raw.githubusercontent.com/Uttam-Nepal/"
       "machine-learning-explained/main/"
       "evaluating-classifiers/transactions.csv")
df = pd.read_csv(url)
print(df.shape)
print(df["fraud"].value_counts().to_dict())
df.head()

### Split, keeping the same fraud rate
- `stratify=y`: keep the same share of fraud in the training and test sets
- `y_test.sum()`: the number of fraud cases in the test set

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns="fraud")
y = df["fraud"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y)
print("test transactions:", len(y_test), " fraud:", y_test.sum())

## 2. The accuracy trap
**Accuracy** = the share of predictions that are right. With rare events, it can be misleading.

### A model that never says fraud
- `DummyClassifier(strategy="most_frequent")`: always predicts the most common answer: not fraud
- `the output`: about 97% accurate, and it never catches a single fraud

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

lazy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print("lazy accuracy:", round(accuracy_score(y_test, lazy.predict(X_test)), 3))

### A real model
- `LogisticRegression`: a model that gives each transaction a fraud probability; predict uses 0.5 as the cut-off
- `the output`: about the same accuracy as the lazy model. So is it any better?

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = make_pipeline(StandardScaler(), LogisticRegression())
model.fit(X_train, y_train)
pred = model.predict(X_test)
print("model accuracy:", round(accuracy_score(y_test, pred), 3))

## 3. The confusion matrix, precision and recall
**Precision** = TP / (TP + FP): of the transactions we flagged, how many were fraud. **Recall** = TP / (TP + FN): of all the fraud, how much we caught.

### Confusion matrix
- `confusion_matrix`: rows: what really happened (0, 1); columns: what we predicted (0, 1)
- `cm.ravel()`: flatten into TN, FP, FN, TP
- `FN`: fraud we missed: most of them!

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, pred)
print(cm)
tn, fp, fn, tp = cm.ravel()
print(f"TN={tn}  FP={fp}  FN={fn}  TP={tp}")

### Precision, recall and F1
- `precision`: when we say fraud, how often we're right
- `recall`: of all the fraud, the share we caught: tiny
- `F1`: one number that balances both: 2 x P x R / (P + R)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

print("precision:", round(precision_score(y_test, pred), 2))
print("recall:   ", round(recall_score(y_test, pred), 2))
print("F1:       ", round(f1_score(y_test, pred), 2))

## 4. Moving the threshold
The model gives a probability. Lowering the cut-off flags more transactions: recall goes up, precision goes down.

### Three thresholds
- `predict_proba(X_test)[:, 1]`: the fraud probability for every test transaction
- `(proba >= t)`: flag it if the probability is at least t
- `the output`: a lower threshold catches more fraud, but raises more false alarms

In [ ]:
proba = model.predict_proba(X_test)[:, 1]
for t in [0.5, 0.2, 0.1]:
    p = (proba >= t).astype(int)
    print(f"threshold {t}: flagged {p.sum():3}  "
          f"precision {precision_score(y_test, p):.2f}  "
          f"recall {recall_score(y_test, p):.2f}")

### The confusion matrix at threshold 0.1
- `ConfusionMatrixDisplay.from_predictions`: draws the confusion matrix as a coloured grid
- `display_labels`: names instead of 0 and 1

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(
    y_test, (proba >= 0.1).astype(int),
    display_labels=["normal", "fraud"], cmap="Blues")
plt.title("threshold 0.1")
plt.show()

## 5. The ROC curve and AUC
The **ROC curve** shows recall against the false alarm rate, FP / (FP + TN), for every threshold. **AUC** is the area under it: 1 is perfect, 0.5 is guessing.

### ROC curve
- `roc_curve`: the false alarm rate and recall at every possible threshold
- `roc_auc_score`: the area under that curve
- `the dashed line`: what random guessing would give

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

fpr, tpr, _ = roc_curve(y_test, proba)
auc = roc_auc_score(y_test, proba)
plt.plot(fpr, tpr, label=f"model (AUC = {auc:.2f})")
plt.plot([0, 1], [0, 1], "--", color="gray", label="random guessing")
plt.xlabel("false alarm rate"); plt.ylabel("recall (fraud caught)")
plt.legend(); plt.show()

## Try it yourself
1. Try a threshold of 0.05. What happens to precision and recall?
2. Work out precision and recall by hand from the confusion matrix at threshold 0.2.
3. Which threshold gives the highest F1? Try 0.1 to 0.5 in steps of 0.05.